# **Naive Bayes Classification**

## **Topic Roadmap**

**1. Global Configuration and Imports**

**2. Dataset Loading**

**3. Data Preparation**

**4. Calculating Prior Probabilities**

**5. Generating Lookup Tables (Conditional Probabilities)**

**6. Predicting a Target Instance**

### **1. Global Configuration and Imports**

Import the essential data manipulation libraries. The incomplete `sklearn.preprocessing` import from the original environment has been removed, as this notebook demonstrates the manual calculation of Naive Bayes probabilities.

In [1]:
import numpy as np
import pandas as pd

### **2. Dataset Loading**

Load the `play_tennis.csv` dataset, which records weather conditions and whether a tennis match was played.

In [ ]:
df = pd.read_csv('docs/Lecture-062-play_tennis.csv')
df.head()

,day,outlook,temp,humidity,wind,play
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes


### **3. Data Preparation**

Remove non-informative features to prevent them from interfering with probability calculations. The `day` column serves merely as an identifier and contains no predictive value.

In [3]:
tennis_df = df.drop(columns=['day']).copy()
tennis_df.head()

,outlook,temp,humidity,wind,play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


### **4. Calculating Prior Probabilities**

Calculate the base probability of each target class occurring before looking at any evidence. In this dataset, there are 9 positive instances and 5 negative instances.

In [4]:
# View class distributions
print(tennis_df['play'].value_counts())

# Calculate Prior Probabilities P(Yes) and P(No)
p_yes = 9 / 14
p_no = 5 / 14

play
Yes    9
No     5
Name: count, dtype: int64


### **5. Generating Lookup Tables (Conditional Probabilities)**

During the training phase, Naive Bayes creates lookup tables detailing the frequencies of every feature combination relative to the target classes. We can use pandas crosstabs to construct these tables.

In [5]:
outlook_crosstab = pd.crosstab(tennis_df['outlook'], tennis_df['play'])
temp_crosstab = pd.crosstab(tennis_df['temp'], tennis_df['play'])
humidity_crosstab = pd.crosstab(tennis_df['humidity'], tennis_df['play'])
wind_crosstab = pd.crosstab(tennis_df['wind'], tennis_df['play'])

display("Outlook:", outlook_crosstab)
display("Temperature:", temp_crosstab)
display("Humidity:", humidity_crosstab)
display("Wind:", wind_crosstab)

'Outlook:'

play,No,Yes
outlook,,
Overcast,0,4
Rain,2,3
Sunny,3,2


'Temperature:'

play,No,Yes
temp,,
Cool,1,3
Hot,2,2
Mild,2,4


'Humidity:'

play,No,Yes
humidity,,
High,4,3
Normal,1,6


'Wind:'

play,No,Yes
wind,,
Strong,3,3
Weak,2,6


### **6. Predicting a Target Instance**

We will evaluate whether a match will be played given the following query conditions: 
**Outlook = Sunny, Temp = Hot, Humidity = High, Wind = Weak**.

Extract the individual conditional probabilities required for this specific instance from the lookup tables.

In [6]:
# Likelihoods for 'Yes'
p_sunny_yes = 2 / 9
p_hot_yes = 2 / 9
p_high_yes = 3 / 9
p_weak_yes = 6 / 9

# Likelihoods for 'No'
p_sunny_no = 3 / 5
p_hot_no = 2 / 5
p_high_no = 4 / 5
p_weak_no = 2 / 5

Multiply the conditional probabilities together along with the prior probability for each class to compute the final scores. The class with the higher resulting score represents the model's prediction.

In [7]:
prob_yes = p_sunny_yes * p_hot_yes * p_high_yes * p_weak_yes * p_yes
prob_no = p_sunny_no * p_hot_no * p_high_no * p_weak_no * p_no

print(f"Probability Score (Yes): {prob_yes:.5f}")
print(f"Probability Score (No): {prob_no:.5f}")

if prob_yes > prob_no:
    print("\nPrediction: Match will be Played")
else:
    print("\nPrediction: Match will not be Played")

Probability Score (Yes): 0.00705
Probability Score (No): 0.02743

Prediction: Match will not be Played


### **Key Revision Notes**

- **The "Naive" Assumption:** The algorithm assumes that every feature is completely independent of the others given the class label. This simplifies calculations so you can just multiply probabilities together, bypassing complex joint distributions.
- **Prior Probability:** The initial probability of an event (e.g., historical rate of playing tennis) before accounting for new evidence.
- **Likelihood:** The probability of seeing a specific feature given a specific class outcome (e.g., how often is it Sunny when a match is *not* played).
- **Training Phase Mechanism:** Under the hood, training a Naive Bayes classifier consists almost entirely of frequency counting to construct the conditional probability lookup tables demonstrated above.